## Escenario 3: MLflow en AWS (EC2 + RDS Postgres + S3)

Este escenario simula un entorno “tipo producción” para un equipo distribuido:

- **Tracking server**: MLflow Server en una instancia **EC2**
- **Backend store (metadata)**: **Amazon RDS PostgreSQL**
- **Artifact store (modelos, plots, datasets, etc.)**: **Amazon S3**
- **Model Registry**: habilitado (depende del backend store)

### Arquitectura (alto nivel)

- Tu laptop / notebook (cliente MLflow)
  - `mlflow.set_tracking_uri("http://<EC2_PUBLIC_DNS>:5000")`
- EC2 (MLflow server)
  - Conectado a RDS (Postgres) para guardar runs/experiments/registry
  - Escribe artifacts en S3

### Seguridad (nivel curso)

- **Sin HTTPS** (solo HTTP) y **sin autenticación** en MLflow
- Security Group en EC2:
  - Puerto `5000` abierto **solo a tu IP**
  - SSH `22` abierto **solo a tu IP**
- RDS:
  - Puerto `5432` accesible **solo desde el Security Group de EC2**

> Importante: para un entorno real se recomienda HTTPS + auth (por ejemplo detrás de un ALB/Nginx + IAM/OIDC) y secretos en AWS Secrets Manager.

## Requisitos previos

### 1) AWS

- Cuenta AWS y permisos para crear recursos (EC2, RDS, S3, IAM)
- Un *key pair* para entrar a EC2 por SSH
- (Recomendado) usar una **VPC** por defecto y subnets públicas/privadas

### 2) Local (tu laptop)

- Python + entorno con `mlflow` y `scikit-learn`
- Acceso a internet a `http://<EC2_PUBLIC_DNS>:5000`
- Si vas a usar credenciales locales para S3 (no recomendado en este escenario): AWS CLI configurado (`aws configure`)

### 3) Decisiones de naming (ejemplo)

- Bucket S3: `mlflow-artifacts-<tu-sufijo>`
- DB name: `mlflow`
- DB user: `mlflow_user`
- Modelo en Registry: `iris-classifier`
"""

## Paso a paso (AWS)

### Paso A — Crear bucket S3 (artifact store)

- Crea un bucket (misma región donde estará EC2/RDS)
- Bloquea acceso público

Ejemplo de estructura que MLflow creará:

- `s3://<BUCKET>/mlflow/<experiment_id>/<run_id>/artifacts/...`

### Paso B — Crear RDS PostgreSQL (backend store)

- Engine: PostgreSQL (versión compatible)
- DB instance: `db.t3.micro` (para curso)
- Storage: mínimo
- **Public access**: No
- Security Group: permite `5432` **solo** desde SG de EC2
- Guarda:
  - Endpoint (host)
  - Puerto
  - Usuario/contraseña
  - Nombre de DB

### Paso C — Crear EC2 (MLflow server)

- AMI: Ubuntu 22.04 (o Amazon Linux)
- Instance type: `t3.small` (curso)
- Security Group:
  - Inbound SSH `22` desde tu IP
  - Inbound `5000` desde tu IP
  - Outbound: default

### Paso D — IAM (acceso a S3)

Asigna a la EC2 un **IAM Role** (Instance Profile) con permisos mínimos al bucket:
- `s3:ListBucket` en el bucket
- `s3:GetObject`, `s3:PutObject`, `s3:DeleteObject` en `bucket/*`

> Recomendación: evita usar `AWS_ACCESS_KEY_ID`/`AWS_SECRET_ACCESS_KEY` en EC2; usa IAM Role.
"""

In [ ]:
"""## Configurar MLflow Server en EC2

Conéctate por SSH a tu instancia EC2 y ejecuta (Ubuntu):

```bash
sudo apt-get update
sudo apt-get install -y python3-pip python3-venv

python3 -m venv mlflow-venv
source mlflow-venv/bin/activate

pip install --upgrade pip
pip install mlflow psycopg2-binary boto3
```

Define variables de entorno (en la misma sesión):

```bash
export AWS_REGION="<REGION>"
export MLFLOW_S3_BUCKET="mlflow-artifacts-<tu-sufijo>"

# RDS
export DB_USER="mlflow_user"
export DB_PASSWORD="<PASSWORD>"
export DB_HOST="<RDS_ENDPOINT>"
export DB_PORT="5432"
export DB_NAME="mlflow"

export BACKEND_STORE_URI="postgresql+psycopg2://${DB_USER}:${DB_PASSWORD}@${DB_HOST}:${DB_PORT}/${DB_NAME}"
export ARTIFACT_ROOT="s3://${MLFLOW_S3_BUCKET}/mlflow"
```

Levanta el server:

```bash
mlflow server \
  --host 0.0.0.0 \
  --port 5000 \
  --backend-store-uri "${BACKEND_STORE_URI}" \
  --default-artifact-root "${ARTIFACT_ROOT}"
```

Abre en tu navegador:

- `http://<EC2_PUBLIC_DNS>:5000`
"""

In [ ]:
"""## Cliente (tu laptop): configurar Tracking URI

Completa `TRACKING_URI` con el DNS público de la EC2:

- Ejemplo: `http://ec2-xx-yy-zz.compute-1.amazonaws.com:5000`
"""

In [ ]:
import os

import mlflow

TRACKING_URI = os.environ.get("MLFLOW_TRACKING_URI", "")  # e.g. http://<EC2_PUBLIC_DNS>:5000
assert TRACKING_URI, "Set MLFLOW_TRACKING_URI or edit TRACKING_URI with your EC2 public DNS"

mlflow.set_tracking_uri(TRACKING_URI)
print(f"tracking URI: '{mlflow.get_tracking_uri()}'")

mlflow.search_experiments()

In [ ]:
import pandas as pd

from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

mlflow.set_experiment("iris-aws-ec2-rds-s3")

with mlflow.start_run(run_name="logreg_baseline"):
    X, y = load_iris(return_X_y=True)

    params = {"C": 0.3, "random_state": 42, "max_iter": 200}
    mlflow.log_params(params)

    mlflow.set_tags(
        {
            "scenario": "3_aws_ec2_rds_s3",
            "developer": "Maria Durango",
            "module": "mlops_tracking",
            "model_family": "logistic_regression",
            "dataset": "iris",
        }
    )

    lr = LogisticRegression(**params).fit(X, y)
    y_pred = lr.predict(X)

    mlflow.log_metric("accuracy", float(accuracy_score(y, y_pred)))
    mlflow.log_metric("f1_macro", float(f1_score(y, y_pred, average="macro")))

    preds_path = "predictions_logreg.csv"
    pd.DataFrame({"y_true": y, "y_pred": y_pred}).to_csv(preds_path, index=False)
    mlflow.log_artifact(preds_path)

    model_info = mlflow.sklearn.log_model(
        lr,
        name="model",
        input_example=X[[0]],
        registered_model_name="iris-classifier",
    )

    print({"model_uri": model_info.model_uri, "artifact_uri": mlflow.get_artifact_uri()})

In [ ]:
"""## Comparar runs (desde el cliente)

Esto te ayuda a ver rápidamente qué configuración/algoritmo está funcionando mejor.
"""

experiment = mlflow.get_experiment_by_name("iris-aws-ec2-rds-s3")
assert experiment is not None

runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["metrics.f1_macro DESC"],
)

runs[[
    "run_id",
    "status",
    "metrics.accuracy",
    "metrics.f1_macro",
    "tags.model_family",
    "tags.scenario",
]]

In [ ]:
"""## Model Registry (AWS): aliases para consumo estable

En un entorno real, es común consumir un modelo por un alias estable, por ejemplo:

- `models:/iris-classifier@champion`
- `models:/iris-classifier@candidate`

Así no dependes de números de versión en tu código.
"""

In [ ]:
from mlflow.tracking import MlflowClient

client = MlflowClient(tracking_uri=mlflow.get_tracking_uri())
model_name = "iris-classifier"

versions = client.search_model_versions(f"name='{model_name}'")
[(v.version, v.run_id, v.current_stage) for v in versions]

In [ ]:
latest = max(versions, key=lambda v: int(v.creation_timestamp))
latest_version = latest.version

client.set_registered_model_alias(model_name, "candidate", latest_version)
client.set_registered_model_alias(model_name, "champion", latest_version)

client.get_registered_model(model_name).aliases

## Troubleshooting rápido

- **No carga la UI**:
  - Revisa SG de EC2 (inbound 5000 desde tu IP)
  - Revisa que `mlflow server` esté corriendo en EC2 y escuchando en `0.0.0.0:5000`
- **Errores conectando a RDS**:
  - SG de RDS debe permitir `5432` desde SG de EC2
  - Verifica endpoint/credenciales y que RDS esté en estado *available*
- **Errores con S3**:
  - Confirma que EC2 tenga IAM Role con permisos al bucket
  - Revisa región del bucket y variables

---

## Diferencias vs escenarios 1 y 2

- Escenario 1: `file store` local (didáctico, no colaborativo)
- Escenario 2: server local + SQLite + artifacts local (colaboración en red local)
- Escenario 3: server remoto + Postgres + S3 (colaboración global y arquitectura escalable)
"""